In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import datasets

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
dataset = datasets.load_dataset("google/code_x_glue_cc_code_refinement", "medium").shuffle()

In [4]:
def letter_tokenizer(s: str):
    return list(s) + ["<EOS>"]

def word_tokenizer(s: str):
    return s.split() + ["<EOS>"]

vocab = set()
def map_fn(x: dict[str, list[str]]):
    # x: {"buggy": [str], "fixed": [str]}
    vocab.update("".join(x["buggy"]) + "".join(x["fixed"]))
    return {
        "buggy": [letter_tokenizer(f) for f in x["buggy"]],
        "fixed": [letter_tokenizer(f) for f in x["fixed"]],
    }

dataset = dataset.map(map_fn, batched=True)
vocab.add("<EOS>")
vocab.add("<PAD>")  # Add padding token to the vocabulary
vocab.add("<SOS>")  # Add start-of-sequence token to the vocabulary

print(dataset["train"][0])

Map:   0%|          | 0/52364 [00:00<?, ? examples/s]

Map:   0%|          | 0/6546 [00:00<?, ? examples/s]

Map:   0%|          | 0/6545 [00:00<?, ? examples/s]

{'id': 4573, 'buggy': ['p', 'u', 'b', 'l', 'i', 'c', ' ', 'v', 'o', 'i', 'd', ' ', 'M', 'E', 'T', 'H', 'O', 'D', '_', '1', ' ', '(', ' ', ')', ' ', '{', ' ', 'T', 'Y', 'P', 'E', '_', '1', ' ', 'V', 'A', 'R', '_', '1', ' ', ';', ' ', 'V', 'A', 'R', '_', '1', ' ', '=', ' ', 'n', 'e', 'w', ' ', 'T', 'Y', 'P', 'E', '_', '1', ' ', '(', ' ', 'T', 'Y', 'P', 'E', '_', '2', ' ', '.', ' ', 'M', 'E', 'T', 'H', 'O', 'D', '_', '2', ' ', '(', ' ', 'V', 'A', 'R', '_', '2', ' ', ')', ' ', ',', ' ', 'V', 'A', 'R', '_', '3', ' ', ')', ' ', ';', ' ', 'V', 'A', 'R', '_', '1', ' ', '=', ' ', 'n', 'e', 'w', ' ', 'T', 'Y', 'P', 'E', '_', '1', ' ', '(', ' ', 'T', 'Y', 'P', 'E', '_', '2', ' ', '.', ' ', 'M', 'E', 'T', 'H', 'O', 'D', '_', '2', ' ', '(', ' ', 'V', 'A', 'R', '_', '4', ' ', ')', ' ', ',', ' ', 'V', 'A', 'R', '_', '5', ' ', ')', ' ', ';', ' ', 'V', 'A', 'R', '_', '1', ' ', '=', ' ', 'n', 'e', 'w', ' ', 'T', 'Y', 'P', 'E', '_', '1', ' ', '(', ' ', 'T', 'Y', 'P', 'E', '_', '2', ' ', '.', ' ', 'M', 'E

In [5]:
vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")
print(f"Vocab: {vocab}")

Vocab size: 89
Vocab: {'t', 'd', 'm', '?', 'W', ' ', ';', 'x', 'V', 'f', 'i', 'e', '2', '&', '%', '}', 'u', '0', '*', '4', 'c', 'g', '<SOS>', 'D', '\n', '=', '8', '/', 'O', 'a', 'y', 'o', 'E', 'k', ':', '1', 'z', 'N', '\\', '3', 'T', 'C', 'I', '{', '+', 'A', 'M', 'H', 'n', '^', 'S', '[', '"', ')', '7', 'U', '<PAD>', 'l', ',', 'B', 'R', '<EOS>', 'b', '_', 'h', '~', 'q', '<', 'F', 'r', '(', '!', '.', 's', '6', 'G', '|', '>', 'v', '9', ']', 'p', '5', 'w', 'P', 'j', '-', 'Y', 'L'}


In [6]:
OHE = {c: i for i, c in enumerate(vocab)}
def ohe_map_fn(x: dict[str, list[str]]):
    return {
        "buggy": [[OHE[c] for c in f]  for f in x["buggy"]],
        "fixed": [[OHE[c] for c in f]  for f in x["fixed"]],
    }

dataset = dataset.map(ohe_map_fn, batched=True, batch_size=5000, remove_columns=["id"])

Map:   0%|          | 0/52364 [00:00<?, ? examples/s]

Map:   0%|          | 0/6546 [00:00<?, ? examples/s]

Map:   0%|          | 0/6545 [00:00<?, ? examples/s]

In [7]:
print(dataset["train"][0])

{'buggy': [81, 16, 62, 57, 10, 20, 5, 78, 31, 10, 1, 5, 46, 32, 40, 47, 28, 23, 63, 35, 5, 70, 5, 53, 5, 43, 5, 40, 87, 84, 32, 63, 35, 5, 8, 45, 60, 63, 35, 5, 6, 5, 8, 45, 60, 63, 35, 5, 25, 5, 48, 11, 83, 5, 40, 87, 84, 32, 63, 35, 5, 70, 5, 40, 87, 84, 32, 63, 12, 5, 72, 5, 46, 32, 40, 47, 28, 23, 63, 12, 5, 70, 5, 8, 45, 60, 63, 12, 5, 53, 5, 58, 5, 8, 45, 60, 63, 39, 5, 53, 5, 6, 5, 8, 45, 60, 63, 35, 5, 25, 5, 48, 11, 83, 5, 40, 87, 84, 32, 63, 35, 5, 70, 5, 40, 87, 84, 32, 63, 12, 5, 72, 5, 46, 32, 40, 47, 28, 23, 63, 12, 5, 70, 5, 8, 45, 60, 63, 19, 5, 53, 5, 58, 5, 8, 45, 60, 63, 82, 5, 53, 5, 6, 5, 8, 45, 60, 63, 35, 5, 25, 5, 48, 11, 83, 5, 40, 87, 84, 32, 63, 35, 5, 70, 5, 40, 87, 84, 32, 63, 12, 5, 72, 5, 46, 32, 40, 47, 28, 23, 63, 12, 5, 70, 5, 8, 45, 60, 63, 74, 5, 53, 5, 58, 5, 8, 45, 60, 63, 54, 5, 53, 5, 6, 5, 15, 24, 61], 'fixed': [81, 16, 62, 57, 10, 20, 5, 78, 31, 10, 1, 5, 46, 32, 40, 47, 28, 23, 63, 35, 5, 70, 5, 53, 5, 43, 5, 40, 87, 84, 32, 63, 35, 5, 8, 45, 

In [8]:
def custom_collate_fn(batch):
    buggy = [torch.tensor(item["buggy"], dtype=torch.long) for item in batch]
    fixed = [torch.tensor(item["fixed"], dtype=torch.long) for item in batch]
    return torch.nn.utils.rnn.pad_sequence(buggy, batch_first=True, padding_value=OHE["<PAD>"]), torch.nn.utils.rnn.pad_sequence(fixed, batch_first=True, padding_value=OHE["<PAD>"])

train_dataloader, val_dataloader, test_dataloader = [
    torch.utils.data.DataLoader(
        dataset[split], batch_size=32, shuffle=True, collate_fn=custom_collate_fn
    ) for split in ["train", "validation", "test"]
]

print(f"Train: {len(train_dataloader.dataset)}")
print(next(iter(train_dataloader)))

Train: 52364
(tensor([[81, 69, 10,  ..., 56, 56, 56],
        [81, 16, 62,  ..., 56, 56, 56],
        [81, 16, 62,  ..., 56, 56, 56],
        ...,
        [81, 16, 62,  ..., 56, 56, 56],
        [81, 69, 10,  ..., 56, 56, 56],
        [81, 16, 62,  ..., 56, 56, 56]]), tensor([[81, 69, 10,  ..., 56, 56, 56],
        [81, 16, 62,  ..., 56, 56, 56],
        [81, 16, 62,  ..., 56, 56, 56],
        ...,
        [81, 16, 62,  ..., 56, 56, 56],
        [81, 69, 10,  ..., 56, 56, 56],
        [81, 16, 62,  ..., 56, 56, 56]]))


In [9]:
class LSTMModel(torch.nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int, hidden_dim: int):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embedding_dim, padding_idx=OHE["<PAD>"])
        self.encoder_lstm = torch.nn.LSTM(embedding_dim, hidden_dim, batch_first=True, num_layers=3)
        self.decoder_lstm = torch.nn.LSTM(embedding_dim, hidden_dim, batch_first=True, num_layers=3)
        self.classifier_mlp = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, vocab_size)  # We don't want to predict the <PAD> token, so we reduce the output size by 1
        )

    def forward(self, x: torch.Tensor, y: torch.Tensor):
        inp_embedded = self.embedding(x)
        # We need to shift y to the right by one position and prepend a <SOS> token at the beginning for the decoder input
        sos_token = torch.full((y.size(0), 1), OHE["<SOS>"], dtype=torch.long, device=y.device)
        _, (h_n, c_n) = self.encoder_lstm(inp_embedded)
        # _ has output features (h_t) from the last layer of the LSTM for each timestep
        # h_n has the hidden state for the last timestep for each layer
        # c_n has the cell state for the last timestep for each layer
        # h_n, c_n are the context vectors from the all the lstm layers

        y_inp = torch.cat((sos_token, y), dim=1)
        y_embedded = self.embedding(y_inp)
        output, _ = self.decoder_lstm(y_embedded, (h_n, c_n))
        # We need output features (h_t) from the last layer of the LSTM for each timestep
        mlp_output = self.classifier_mlp(output[:, :-1])

        return mlp_output

model = LSTMModel(vocab_size, embedding_dim=256, hidden_dim=512).to(DEVICE)

In [10]:
def train(model, train_dataloader, valid_dataloader, epochs=10):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = torch.nn.CrossEntropyLoss(ignore_index=OHE["<PAD>"])  # Ignore the <PAD> token in the loss calculation

    losses = []
    valid_losses = []
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_dataloader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(X_batch, y_batch)
            loss = criterion(outputs.permute(0, 2, 1), y_batch)  # crossentropy expects (N, C, d1, d2, ...) for input and (N, d1, d2, ...) for target
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        losses.append(total_loss / len(train_dataloader))
        print(f"Epoch {epoch+1}/{epochs}, Loss: {losses[-1]:.4f}")

        model.eval()
        with torch.no_grad():
            valid_loss = 0
            for X_batch, y_batch in valid_dataloader:
                X_batch = X_batch.to(DEVICE)
                y_batch = y_batch.to(DEVICE)
                outputs = model(X_batch, y_batch)
                loss = criterion(outputs.permute(0, 2, 1), y_batch)
                valid_loss += loss.item()
        valid_losses.append(valid_loss / len(valid_dataloader))
        print(f"Validation Loss: {valid_losses[-1]:.4f}")

    return losses, valid_losses

In [11]:
train_losses, valid_losses = train(model, train_dataloader, val_dataloader, epochs=12)
torch.save(model.state_dict(), 'models/lstm.pth')
plt.plot(train_losses, label='Train Loss')
plt.plot(valid_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

Epoch 1/12, Loss: 0.5599
Validation Loss: 0.3364


KeyboardInterrupt: 